In [0]:
from pyspark.sql.functions import col , when , sum , count ,current_timestamp , max

In [0]:
spark.sql("""
          SHOW SCHEMAS IN globalretail
          """).show(truncate=False)

In [0]:
source_catalog = "globalretail_snowflake_conn_catalog"

display(
    spark.sql(f"SHOW SCHEMAS IN {source_catalog}")
)
display(
    spark.sql(f"SHOW TABLES IN {source_catalog}.RAW")
)

Loading Customer Data from snowflake

In [0]:
source_catalog="globalretail_snowflake_conn_catalog"

source_table = f"{source_catalog}.raw.customers_raw"

target_table = "globalretail.bronze.customers_raw"

watermark_table = "globalretail.validation.bronze_source_watermarks"


hwm_row = spark.table(watermark_table).filter(col("entity_name") == "customers_raw").first()

last_hwm = hwm_row["last_source_updated_at"]




source_df = spark.table(source_table).filter(col("UPDATED_AT") > last_hwm)

if source_df.isEmpty():
    print(f"No new records found in Snowflake after {last_hwm}.")

else:
    new_hwm = source_df.select(max("UPDATED_AT")).collect()[0][0]

    incremental_df= source_df.withColumn("ingested_at", current_timestamp())

    incremental_df = incremental_df.select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone",
        "address_line1",
        "city",
        "state",
        "country_code",
        "postal_code",
        "loyalty_tier",
        "marketing_opt_in",
        "source_system",
        "updated_at",
        "is_active",
        "ingested_at"
    )

    incremental_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

    print("Initial customer load completed!")

    spark.sql(f"""
              UPDATE {watermark_table}
              SET last_source_updated_at = '{new_hwm}'
              WHERE entity_name = 'customers_raw'

              """)

    print(f"Successfully loaded new data. HWM updated to: {new_hwm}")



In [0]:
bronze_df = spark.table(
    "globalretail.bronze.customers_raw"
)

print("Total Bronze records:", bronze_df.count())

display(bronze_df.limit(10))

In [0]:
source_count = spark.table(
    "globalretail_snowflake_conn_catalog.raw.customers_raw"
).count()

bronze_count = spark.table(
    "globalretail.bronze.customers_raw"
).count()

print("Snowflake customer RAW:", source_count)
print("Databricks customer Bronze:", bronze_count)
print("Counts match:", source_count == bronze_count)

In [0]:
source_catalog="globalretail_snowflake_conn_catalog"

source_table = f"{source_catalog}.raw.stores_raw"

target_table = "globalretail.bronze.stores_raw"

watermark_table = "globalretail.validation.bronze_source_watermarks"


hwm_rows = spark.table(watermark_table).filter(col("entity_name")=="stores_raw").first()

last_hwm=hwm_rows["last_source_updated_at"]

print(hwm_rows)
print(last_hwm)


source_df = spark.table(source_table).filter(col("UPDATED_AT") > last_hwm)

if source_df.isEmpty():
    print(f"No new records found in Snowflake after {last_hwm}.")

else:
    new_hwm = source_df.select(max("UPDATED_AT")).collect()[0][0]
    print(f"New watermark: {new_hwm}")
    

    incremental_df = source_df.withColumn("ingested_at",current_timestamp())

    incremental_df = incremental_df.select(

        "store_id",
        "store_name",
        "city",
        "country_code",
        "timezone_name",
        "region",
        "opened_date",
        "updated_at",
        "is_active",
        "ingested_at"
    )

    incremental_df.write\
        .format("delta")\
        .mode("append")\
        .saveAsTable(target_table)

    print("Initial store load completed")


    spark.sql(f"""
              UPDATE globalretail.validation.bronze_source_watermarks
              SET last_source_updated_at = '{new_hwm}'
              WHERE entity_name = 'stores_raw'
              
              
              """)
    print(f"Successfully loaded new  stores data. HWM updated to: {new_hwm}")

In [0]:
bronze_stores_df = spark.table(
    "globalretail.bronze.stores_raw"
)

print("Total Bronze stores records:", bronze_df.count())

display(bronze_stores_df.limit(10))

In [0]:
source_catalog="globalretail_snowflake_conn_catalog"
source_table = f"{source_catalog}.raw.products_raw"
target_table = "globalretail.bronze.products_raw"
watermark_table = "globalretail.validation.bronze_source_watermarks"


hwm_row_products = spark.table(watermark_table).filter(col("entity_name")=="products_raw").first()
print(hwm_row_products)

last_hwm=hwm_row_products["last_source_updated_at"]

print(last_hwm)

source_df = spark.table(source_table).filter(col("updated_at")>last_hwm)

if source_df.isEmpty():
    print(f"No new records found in Snowflake after {last_hwm}.")
else:
    new_hwm = source_df.select(max("UPDATED_AT")).collect()[0][0]
    print(f"New watermark: {new_hwm}")
    
    incremental_products_df = source_df.withColumn("ingested_at",current_timestamp())

    incremental_products_df=incremental_products_df.select(
        "product_id",
        "sku",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "unit_price",
        "unit_cost",
        "currency_code",
        "reorder_point",
        "is_active",
        "updated_at",
        "ingested_at"
    )


    incremental_products_df.write\
        .format("delta")\
        .mode("append")\
        .saveAsTable(target_table)

    print("Initial product load completed")

    spark.sql(f"""
              UPDATE globalretail.validation.bronze_source_watermarks
              SET last_source_updated_at = '{new_hwm}'
              WHERE entity_name = 'products_raw'
              
              
              """)

    print(f"Successfully loaded new products data. HWM updated to: {new_hwm}")

In [0]:
bronze_products_df = spark.table(
    "globalretail.bronze.products_raw"
)

print("Total Bronze stores records:", bronze_products_df.count())

display(bronze_products_df.limit(10))

In [0]:
source_catalog="globalretail_snowflake_conn_catalog"
source_table = f"{source_catalog}.raw.sales_raw"
target_table = "globalretail.bronze.sales_raw"
watermark_table = "globalretail.validation.bronze_source_watermarks"



hwm_row = spark.table(watermark_table).filter(col("entity_name") == "sales_raw").first()

last_hwm = hwm_row["last_source_updated_at"]

print(last_hwm)


source_df = spark.table(source_table).filter(col("updated_at")>last_hwm)

if source_df.isEmpty():
    print(f"No new Records found in Snoeflake after {last_hwm}")
else:
    new_hwm = source_df.select(max('UPDATED_AT')).collect()[0][0]

    incremental_sales_df = source_df.withColumn("ingested_at",current_timestamp())

    incremental_sales_df=incremental_sales_df.select(
      "order_id",
      "order_line_id",
      "order_ts_utc",
      "customer_id",
      "product_id",
      "store_id",
      "channel",
      "quantity",
      "unit_price",
      "discount_amount",
      "tax_amount",
      "currency_code",
      "payment_status",
      "updated_at",
      "ingested_at"
    )

    incremental_sales_df.write\
        .format("delta")\
        .mode("append")\
        .saveAsTable(target_table)

    print("Initial sales load completed")

    spark.sql(f"""
              UPDATE globalretail.validation.bronze_source_watermarks
              SET last_source_updated_at = '{new_hwm}'
              WHERE entity_name = 'sales_raw'

              """)
    print(f"Successfully loaded new sales data. HWM updated to: {new_hwm}")


